# Week 5 Deliverable: Bioinformatics Pipeline

## Overview
This notebook implements a complete bioinformatics pipeline for variant calling in CYP genes.

### Genes of Interest
- CYP2C8: chr10:95,036,772-95,069,497
- CYP2C9: chr10:94,938,658-94,990,091
- CYP2C19: chr10:94,762,681-94,855,547

All three genes are located on chromosome 10.


## Step 0: Download Sequencing Data

Download Illumina short-read and PacBio long-read samples.


In [ ]:
%%bash
# Create directories
mkdir -p data results

# Download Illumina short-read data (interleaved paired-end FASTQ)
if [ ! -f data/illumina.fq ]; then
    echo "Downloading Illumina data..."
    wget -qO- https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2 | bunzip2 > data/illumina.fq
    echo "Illumina data download complete."
else
    echo "data/illumina.fq already exists."
fi

# Download PacBio long-read data
if [ ! -f data/pacbio.fq ]; then
    echo "Downloading PacBio data..."
    wget -qO- https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2 | bunzip2 > data/pacbio.fq
    echo "PacBio data download complete."
else
    echo "data/pacbio.fq already exists."
fi

# Check downloaded files
echo ""
echo "Data files:"
ls -lh data/*.fq 2>/dev/null || echo "No FASTQ files found"


## Step 1: Download Reference Genome

Download chromosome 10 from hg38 (GRCh38) as reference.


In [ ]:
%%bash
# Download chr10 reference genome
if [ ! -f results/chr10.fa ]; then
    echo "Downloading chr10 reference genome..."
    wget -q -O results/chr10.fa.gz http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
    gunzip results/chr10.fa.gz
    echo "Download complete."
else
    echo "results/chr10.fa already exists."
fi

# Check file size
ls -lh results/chr10.fa


## Step 2: Alignment with minimap2

Align both samples to the reference genome using appropriate parameters for each technology.


In [ ]:
%%bash
# Align Illumina short reads
echo "Aligning Illumina reads..."
minimap2 -ax sr results/chr10.fa data/illumina.fq | samtools view -bS - | samtools sort -o results/illumina.bam
samtools index results/illumina.bam
echo "Illumina alignment complete."

# Align PacBio long reads
echo "Aligning PacBio reads..."
minimap2 -ax map-pb results/chr10.fa data/pacbio.fq | samtools view -bS - | samtools sort -o results/pacbio.bam
samtools index results/pacbio.bam
echo "PacBio alignment complete."

# Check alignment statistics
echo ""
echo "=== Illumina BAM stats ==="
samtools flagstat results/illumina.bam

echo ""
echo "=== PacBio BAM stats ==="
samtools flagstat results/pacbio.bam


## Step 3: Variant Calling

Call variants in the CYP gene regions using bcftools.


In [ ]:
%%bash
# Define regions of interest (CYP genes)
# Note: BAM file uses "chr10" (with chr prefix)
REGIONS="chr10:94761900-94853205,chr10:94938658-94990091,chr10:95036772-95069497"

# Call variants for Illumina
echo "Calling variants for Illumina..."
bcftools mpileup -f results/chr10.fa -r $REGIONS results/illumina.bam | \
    bcftools call -mv -Oz -o results/illumina.vcf.gz
bcftools index results/illumina.vcf.gz

# Call variants for PacBio
echo "Calling variants for PacBio..."
bcftools mpileup -f results/chr10.fa -r $REGIONS results/pacbio.bam | \
    bcftools call -mv -Oz -o results/pacbio.vcf.gz
bcftools index results/pacbio.vcf.gz

echo ""
echo "=== Variant counts ==="
echo "Illumina variants:"
bcftools view -H results/illumina.vcf.gz | wc -l
echo "PacBio variants:"
bcftools view -H results/pacbio.vcf.gz | wc -l


## Step 4: Phasing

Phase variants using HapCUT2 or HapTree-X.


In [ ]:
%%bash
# Phase Illumina variants
echo "Phasing Illumina variants with HapCUT2..."
echo "VCF variants count: $(bcftools view -H results/illumina.vcf.gz | wc -l)"

# Extract haplotype-informative reads
echo "Running extractHAIRS..."
extractHAIRS --bam results/illumina.bam \
    --VCF results/illumina.vcf.gz \
    --out results/illumina_fragments.txt || echo "extractHAIRS failed with exit code $?"

echo "Fragments extracted: $(wc -l < results/illumina_fragments.txt 2>/dev/null || echo 0)"

# Run HapCUT2 for phasing  
echo "Running HAPCUT2..."
HAPCUT2 --fragments results/illumina_fragments.txt \
    --VCF results/illumina.vcf.gz \
    --output results/illumina_phased.hapcut || echo "HAPCUT2 failed with exit code $?"

echo "HapCUT2 output lines: $(wc -l < results/illumina_phased.hapcut 2>/dev/null || echo 0)"

# Only convert if HapCUT2 produced output
if [ -s results/illumina_phased.hapcut ]; then
    echo "Converting to phased VCF..."
    whatshap hapcut2vcf results/illumina_phased.hapcut \
        results/illumina.vcf.gz \
        -o results/illumina_phased.vcf || echo "whatshap conversion failed with exit code $?"
    
    # Compress and index the phased VCF
    bgzip -f results/illumina_phased.vcf
    bcftools index results/illumina_phased.vcf.gz
    echo "Illumina phasing complete."
else
    echo "WARNING: HapCUT2 produced no output, creating empty phased VCF"
    touch results/illumina_phased.vcf
    bgzip -f results/illumina_phased.vcf
fi

# Phase PacBio variants
echo "Phasing PacBio variants with HapCUT2..."

# Extract haplotype-informative reads
extractHAIRS --pacbio 1 \
    --bam results/pacbio.bam \
    --VCF results/pacbio.vcf.gz \
    --out results/pacbio_fragments.txt

# Run HapCUT2 for phasing
HAPCUT2 --fragments results/pacbio_fragments.txt \
    --VCF results/pacbio.vcf.gz \
    --output results/pacbio_phased.hapcut

# Convert HapCUT2 block format to phased VCF using WhatsHap
whatshap hapcut2vcf results/pacbio_phased.hapcut \
    results/pacbio.vcf.gz \
    -o results/pacbio_phased.vcf

# Compress and index the phased VCF
bgzip -f results/pacbio_phased.vcf
bcftools index results/pacbio_phased.vcf.gz

echo "PacBio phasing complete."

# Show phasing statistics
echo ""
echo "=== Phasing results ==="
echo "Illumina phased blocks:"
grep "BLOCK" results/illumina_phased.hapcut | wc -l
echo "PacBio phased blocks:"
grep "BLOCK" results/pacbio_phased.hapcut | wc -l

echo ""
echo "Check phased VCF files:"
ls -lh results/*_phased.vcf.gz


## Step 5: Variant Comparison (Temporarily Disabled)

This step is temporarily disabled while debugging Step 4 phasing issues.


### Discussion: Are discordant variants true variants or sequencing artifacts?

Based on the analysis above, we can evaluate each discordant variant by considering:

1. **Quality Score**: Higher quality scores suggest more confidence in the variant call
2. **Read Depth**: Low depth may indicate insufficient coverage to detect the variant
3. **Technology-specific biases**:
   - Illumina short reads may miss variants in repetitive regions or have PCR/sequencing errors
   - PacBio long reads have higher error rates but can span difficult regions
4. **IGV Inspection** (optional): Visual inspection of BAM files can reveal:
   - Whether reads support the variant call
   - Presence of mapping artifacts
   - Strand bias or other technical issues

Typically, shared variants are more likely to be true variants, while technology-specific variants may be:
- True variants that one technology failed to detect (e.g., in difficult regions)
- Sequencing artifacts specific to that technology
- Alignment errors

## Step 6: Star-Allele Identification

Identify star-alleles using PharmVar database.


In [ ]:
# TODO: Implement star-allele identification
print("Star-allele identification to be implemented")


## Time Estimate

Estimated time to complete this assignment: 8-12 hours

Breakdown:
- Understanding requirements: 1 hour
- Setting up tools and environment: 1 hour
- Downloading and aligning data: 2 hours
- Variant calling and phasing: 2-3 hours
- Variant comparison and analysis: 2-3 hours
- Star-allele identification: 1-2 hours
- Documentation and cleanup: 1 hour
